# Smart Maintenance Assistant — RAG Step by Step

**Retrieval-Augmented Generation (RAG) for Predictive Maintenance Decision Support**

> **Kernel:** Select **Smart Maintenance (venv)** (top-right in Cursor/VS Code).  
> If you see `ModuleNotFoundError`, you are on the wrong Python — not the project `venv`.

This notebook walks through each stage of the RAG pipeline used in this project:

1. **Document parsing** — load PDF, TXT, and Markdown maintenance manuals
2. **Chunking** — split documents into overlapping text chunks
3. **Embeddings** — convert chunks into numerical vectors
4. **Chroma vector database** — store and persist embeddings
5. **Similarity search** — retrieve the most relevant chunks for a question
6. **LLM responses with citations** — generate grounded answers with `[1]`, `[2]`, etc.

> Run cells top-to-bottom. Make sure the virtual environment is active and dependencies are installed (`pip install -r requirements.txt`).

## Setup

In [ ]:
import sys
import subprocess
from pathlib import Path

# --- Environment check (fixes ModuleNotFoundError: pypdf) ---
VENV_MARKER = "smart-maintenance-assistant" + chr(92) + "venv"
print("Python:", sys.executable)
if VENV_MARKER not in sys.executable.replace("/", chr(92)):
    raise RuntimeError(
        "Wrong kernel. Click top-right kernel selector and choose "
        "'Smart Maintenance (venv)' or the venv python at "
        "smart-maintenance-assistant\\venv\\Scripts\\python.exe"
    )

try:
    import pypdf  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf", "-q"])
    import pypdf

# Ensure project root is on the Python path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, CHROMA_DIR, EMBEDDING_MODEL, TOP_K, CHUNK_SIZE, CHUNK_OVERLAP

print(f"Project root: {PROJECT_ROOT}")
print(f"Documents:    {DATA_DIR}")
print(f"Chroma DB:    {CHROMA_DIR}")
print(f"Embedding:    {EMBEDDING_MODEL}")
print("pypdf OK")

## Pipeline Overview

```
Maintenance Docs (PDF/TXT/MD)
        │
        ▼
   1. Document Parsing
        │
        ▼
   2. Chunking (overlap)
        │
        ▼
   3. Embeddings (sentence-transformers)
        │
        ▼
   4. Chroma Vector DB
        │
        ▼
   User Question ──► 5. Similarity Search (top-k)
                          │
                          ▼
                    6. LLM + Citations
```

---
## Step 1: Document Parsing

We read maintenance manuals from `data/documents/`. Supported formats:
- **PDF** — extracted page by page (`pypdf`)
- **TXT / MD** — read as plain text

Each parsed section becomes a `ParsedDocument` with `source`, `text`, and optional `page`.

In [ ]:
from src.parsers import parse_directory, parse_file

# List available documents
doc_files = sorted(DATA_DIR.rglob("*"))
doc_files = [f for f in doc_files if f.is_file()]
print("Files in data/documents/:")
for f in doc_files:
    print(f"  - {f.name}")

In [ ]:
# Parse all documents
documents = parse_directory(DATA_DIR)
print(f"Parsed {len(documents)} document section(s)\n")

# Inspect the first parsed document
sample = documents[0]
print(f"Source: {sample.source}")
print(f"Page:   {sample.page}")
print(f"Length: {len(sample.text)} characters")
print("\n--- Preview (first 400 chars) ---")
print(sample.text[:400])

---
## Step 2: Chunking

Long documents are split into smaller **chunks** so retrieval can find specific passages.

- **Chunk size**: 500 characters (configurable via `CHUNK_SIZE`)
- **Overlap**: 50 characters (`CHUNK_OVERLAP`) — prevents cutting sentences in half at boundaries

Chunks are split on natural separators: paragraph breaks → newlines → sentences → words.

In [ ]:
from src.chunker import chunk_documents

chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"Created {len(chunks)} chunks from {len(documents)} document section(s)")
print(f"Settings: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")

In [ ]:
# Show a few chunks with metadata
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ---")
    print(f"Source:   {chunk.source}")
    print(f"Chunk ID: {chunk.chunk_id}")
    print(f"Length:   {len(chunk.text)} chars")
    print(chunk.text[:200], "...\n")

---
## Step 3: Embeddings

Embeddings turn text into **vectors** (lists of numbers) that capture semantic meaning.

Similar meanings → similar vectors → better retrieval.

We use **`all-MiniLM-L6-v2`** from `sentence-transformers` — runs locally, no API key needed.

In [ ]:
from src.embeddings import embed_texts, embed_query

# Embed the first 3 chunks
sample_texts = [c.text for c in chunks[:3]]
sample_embeddings = embed_texts(sample_texts)

print(f"Number of embeddings: {len(sample_embeddings)}")
print(f"Vector dimension:     {len(sample_embeddings[0])}")
print(f"First 5 values:       {sample_embeddings[0][:5]}")

In [ ]:
# Compare: maintenance question vs random text
query = "What vibration level is critical for Pump-3?"
query_vec = embed_query(query)

import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b)

print(f"Query: {query}\n")
print("Cosine similarity with first 5 chunks:")
for i, emb in enumerate(embed_texts([c.text for c in chunks[:5]])):
    score = cosine_similarity(query_vec, emb)
    print(f"  Chunk {i} ({chunks[i].source[:20]}...): {score:.4f}")

---
## Step 4: Chroma Vector Database

**Chroma** stores chunk text, embeddings, and metadata (source file, page, chunk ID).

Data is persisted in `chroma_db/` so you don't need to re-embed every time.

In [ ]:
from src.vector_store import add_chunks, collection_count, reset_collection, get_collection

# Reset and index all chunks (same as: python ingest.py --reset)
reset_collection()
added = add_chunks(chunks)
total = collection_count()

print(f"Added {added} chunks to Chroma")
print(f"Collection total: {total}")
print(f"Stored at: {CHROMA_DIR}")

In [ ]:
# Peek inside the collection
collection = get_collection()
peek = collection.peek(limit=2)

for i, doc_id in enumerate(peek["ids"]):
    print(f"ID:       {doc_id}")
    print(f"Source:   {peek['metadatas'][i]['source']}")
    print(f"Preview:  {peek['documents'][i][:150]}...\n")

---
## Step 5: Similarity Search

When a user asks a question:
1. Embed the question
2. Find the **top-k** most similar chunks in Chroma (cosine distance)
3. Return chunks ranked by relevance score

In [ ]:
from src.retriever import search

question = "What vibration level is critical for Pump-3?"
results = search(question, top_k=TOP_K)

print(f"Question: {question}\n")
print(f"Retrieved {len(results)} chunks (top_k={TOP_K}):\n")

for r in results:
    loc = f", page {r.page}" if r.page else ""
    print(f"[{r.citation_id}] {r.source}{loc} — relevance: {r.score:.1%}")
    print(f"    {r.text[:180]}...\n")

In [ ]:
# Try another maintenance question
question2 = "When should HVAC Unit-7 filters be replaced?"
results2 = search(question2, top_k=3)

print(f"Question: {question2}\n")
for r in results2:
    print(f"[{r.citation_id}] {r.source} ({r.score:.1%}): {r.text[:120]}...")

---
## Step 6: LLM Responses with Citations

The final RAG step combines **retrieved context** with an **LLM** to produce a natural-language answer.

The LLM is instructed to:
- Answer **only** from the retrieved excerpts
- Cite sources inline as `[1]`, `[2]`, etc.
- Avoid inventing thresholds or procedures

**LLM priority:** OpenAI → Ollama (local) → extractive fallback (no LLM)

In [ ]:
from src.rag import ask, _format_context

# Show how context is formatted for the LLM
citations = search("What indicates high bearing failure risk?", top_k=3)
context = _format_context(citations)

print("Context sent to the LLM:\n")
print(context)

In [ ]:
# Full RAG answer
response = ask("What vibration level is critical for Pump-3?")

print("=== ANSWER ===")
print(response.answer)
print(f"\n=== MODE: {response.mode} ===")
print(f"=== CITATIONS: {len(response.citations)} sources ===")

for c in response.citations:
    print(f"  [{c.citation_id}] {c.source} — {c.score:.1%}")

In [ ]:
# Ask a few more predictive maintenance questions
questions = [
    "When should I replace HVAC Unit-7 filters?",
    "What should I do if motor current exceeds 20A?",
    "What are common root causes of bearing failure?",
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}")
    r = ask(q)
    print(f"A: {r.answer[:500]}{'...' if len(r.answer) > 500 else ''}")
    print(f"   (mode: {r.mode}, sources: {len(r.citations)})\n")

---
## Summary

| Step | Module | What it does |
|------|--------|--------------|
| 1. Parsing | `src/parsers.py` | PDF/TXT/MD → structured text |
| 2. Chunking | `src/chunker.py` | Split text with overlap |
| 3. Embeddings | `src/embeddings.py` | Text → vectors (MiniLM) |
| 4. Chroma | `src/vector_store.py` | Persist vectors + metadata |
| 5. Search | `src/retriever.py` | Top-k similarity retrieval |
| 6. RAG | `src/rag.py` | LLM answer + citations |

### Next steps
- Run the chat UI: `streamlit run app.py`
- Add your own PDFs to `data/documents/` and re-run Step 4
- Set `OPENAI_API_KEY` in `.env` for better natural-language answers
- Tune `CHUNK_SIZE`, `CHUNK_OVERLAP`, and `TOP_K` in `.env`